In [14]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 21, Finished, Available, Finished)

In [12]:
%pip install unittest-xml-reporting

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 18, Finished, Available, Finished)


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [39]:
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Bronze Ingestion Tests") \
    .getOrCreate()

class BronzeIngestionTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id

    def setUp(self):
        self.token = mssparkutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
        self.runtime_context = mssparkutils.runtime.context
        print(self.runtime_context)

    def test_bronze_clinical_ingestion_folder_is_empty(self):

        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/Clinical/FHIR-NDJSON/FHIR-HDS"))
        ingest_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/Clinical/FHIR-NDJSON/FHIR-HDS")
        
        # Assert the sample data folder exists but is empty
        self.assertEqual(1, len(ingest_files_folder))
        sample_data_folder = mssparkutils.fs.ls(ingest_files_folder[0].path)
        self.assertEqual(0, len(sample_data_folder))

    def test_bronze_lakehouse_folder_structure_is_hydrated(self):
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/External"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Failed"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/ReferenceData"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/SampleData"))

    def test_bronze_ingestion_processed_sample_data(self):

        processed_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/Clinical/FHIR-NDJSON/FHIR-HDS/")
        self.assertEqual(1, len(processed_files_folder))

        year_folders = mssparkutils.fs.ls(processed_files_folder[0].path)
        self.assertEqual(1, len(year_folders))

        months_folder = mssparkutils.fs.ls(year_folders[0].path)
        self.assertEqual(1, len(months_folder))

        days_folder = mssparkutils.fs.ls(months_folder[0].path)
        self.assertEqual(1, len(days_folder))
        
        sample_data_folder = mssparkutils.fs.ls(days_folder[0].path)
        self.assertEqual(68, len(sample_data_folder))

        resource_types = {}
        for pf in sample_data_folder:
            file_name = str(pf.path).split("/")[-1]
            resource_type = file_name.split("_", maxsplit=1)[1].split(".")[0].split("-")[0]
            
            if resource_type not in resource_types:
                resource_types[resource_type] = 0
            
            resource_types[resource_type] = resource_types[resource_type] + 1

        expected_counts = {
            "Observation": 6,
            "MedicationRequest": 6,
            "Procedure": 7,
            "Encounter": 7,
            "Condition": 8,
            "ExplanationOfBenefit": 1,
            "Appointment": 1,
            "CarePlan": 1,
            "Goal": 1,
            "Patient": 8,
            "DocumentReference": 2,
            "Location": 2,
            "PractitionerRole": 5,
            "Organization": 5,
            "Practitioner": 5,
            "DiagnosticReport": 1,
            "RiskAssessment": 1,
            "AllergyIntolerance": 1
        }

        for k in resource_types.keys():
            self.assertEqual(expected_counts[k], resource_types[k])

    def test_clincal_fhir_table_is_populated(self):

        expected_clinical_fhir_resource_types = [
            "Encounter",
            "MedicationRequest",
            "Procedure",
            "Condition",
            "Observation",
            "Patient",
            "Appointment",
            "Goal",
            "CarePlan",
            "ExplanationOfBenefit",
            "DocumentReference",
            "DiagnosticReport",
            "PractitionerRole",
            "Organization",
            "Location",
            "Practitioner",
            "AllergyIntolerance"
        ]
        
        df = self.spark.sql("SELECT resourceType, COUNT(*) as count FROM ClinicalFhir GROUP BY resourceType")
        
        # Collect the results
        results = df.collect()
        
        # Print out the counts for verification
        resource_types = set()
        for row in results:
            # print(f"resourceType: {row['resourceType']}, count: {row['count']}")
            resource_types.add(row['resourceType'])

        # Ensure there are non-zero counts for each entry that has a column named "resourceType"
        non_zero_counts = df.filter(df["count"] > 0).count()
        
        for expected_resource_type in expected_clinical_fhir_resource_types:
            self.assertIn(expected_resource_type, resource_types)

        self.assertGreater(non_zero_counts, 0, "There should be non-zero counts for each entry with a resourceType")


def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(BronzeIngestionTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 46, Finished, Available, Finished)


Running tests...
----------------------------------------------------------------------
  test_bronze_clinical_ingestion_folder_is_empty (__main__.BronzeIngestionTests.test_bronze_clinical_ingestion_folder_is_empty) ... ok (6.865s)

----------------------------------------------------------------------
Ran 4 tests in 7.747s

OK

Generating XML reports...


{'currentNotebookName': 'TestRunner', 'currentWorkspaceName': 'HDS_IT_Test_1223_1012', 'defaultLakehouseName': 'healthcare1_msft_bronze', 'defaultLakehouseId': 'e0ebaeb9-d8c6-4ee1-ba6c-afc66d642aeb', 'parentRunId': None, 'isReferenceRun': False, 'defaultLakehouseWorkspaceId': '746620bd-4ecb-4872-96d3-6b3b8ffd0da1', 'hcReplId': None, 'activityId': '68d43252-d1bf-4fef-8fda-abe43343a9e1', 'defaultLakehouseWorkspaceName': 'HDS_IT_Test_1223_1012', 'currentWorkspaceId': '746620bd-4ecb-4872-96d3-6b3b8ffd0da1', 'referenceTreePath': None, 'clusterId': '8b7a4a7a-caa8-4753-aa23-b36d5bcc1ef5', 'poolName': 'Starter Pool', 'environmentId': '', 'currentNotebookId': 'b32a31b8-e510-4007-82c3-ceba4511044f', 'userId': '2870bbf5-1a76-4a57-95bf-a35a0cad5ab6', 'environmentWorkspaceId': '', 'userName': 'Evan Miller', 'currentRunId': None, 'isForPipeline': False, 'rootRunId': None}
